# Notebook 1: Data Curation (AKT1 Bioactivity, ChEMBL)



# Environment Setup

In [ ]:
# Run this cell in Google Colab if required.
# Developed with pandas, numpy, scikit-learn, RDKit, and chembl_webresource_client.
!pip -q install rdkit chembl_webresource_client

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, inchi
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams

from chembl_webresource_client.new_client import new_client

RDLogger.DisableLog("rdApp.*")

print("Package versions (record these for reproducibility):")
print(f"  RDKit: {rdkit.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  NumPy: {np.__version__}")

plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "font.size": 10,
})

activity = new_client.activity
_largest_fragment_remover = rdMolStandardize.LargestFragmentChooser()
_uncharger = rdMolStandardize.Uncharger()

# PAINS A/B/C substructure catalog (used in the nuisance-compound filter below)
_pains_params = FilterCatalogParams()
_pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
_pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_B)
_pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_C)
_pains_catalog = FilterCatalog.FilterCatalog(_pains_params)

# --- Config ---
MIN_ASSAY_CONFIDENCE = 8       # keep assays with confidence_score >= this (9 = direct single protein)
PIC50_IQR_MULTIPLIER = 1.5     # Tukey's-rule multiplier for pIC50 outlier flagging
REMOVE_PIC50_OUTLIERS = False  # if True, drop flagged pIC50 outliers from the saved dataset


# Step 1: Fetch Bioactivity Data from ChEMBL




In [ ]:
def fetch_bioactivity(chembl_id, target_name):
    acts = activity.filter(
        target_chembl_id=chembl_id,
        target_type='SINGLE PROTEIN',
        standard_type='IC50',
        assay_type='B',
        standard_relation='=',
        target_organism='Homo sapiens'
    ).only([
        'molecule_chembl_id',
        'canonical_smiles',
        'standard_value',
        'standard_units',
        'pchembl_value',
        'assay_chembl_id',
        'data_validity_comment'
    ])

    df = pd.DataFrame.from_records(acts)
    df['target'] = target_name
    return df


# Step 2: Drop Missing SMILES / IC50

In [ ]:
def drop_missing(df):
    df = df.dropna(subset=['canonical_smiles', 'standard_value'])
    df = df[df['canonical_smiles'].str.strip() != '']
    df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
    df = df.dropna(subset=['standard_value'])
    df = df[df['standard_value'] > 0]
    print(f"  After dropping missing: {len(df)} rows")
    return df

# Step 3: Keep Only nM/uM/pM/mM Units + Normalize to nM


In [ ]:
def normalize_units(df):
    unit_conversion = {
        'nM': 1,
        'uM': 1000,
        'pM': 0.001,
        'mM': 1_000_000,
    }
    before = len(df)
    df = df[df['standard_units'].isin(unit_conversion.keys())].copy()
    df['IC50_nM'] = df['standard_value'] * df['standard_units'].map(unit_conversion)
    print(f"  After unit filter: {len(df)} rows (dropped {before - len(df)} with non-molar/unrecognized units)")
    return df


# Step 4: Convert IC50 -> pIC50

In [ ]:
def compute_pIC50(df):
    df = df.copy()
    df['pIC50'] = -np.log10(df['IC50_nM'] * 1e-9)   # nM to M, then -log10
    df['pchembl_value'] = pd.to_numeric(df['pchembl_value'], errors='coerce')
    print(f"  pIC50 range: {df['pIC50'].min():.2f} to {df['pIC50'].max():.2f}")
    print(f"  Missing ChEMBL pchembl_value: {df['pchembl_value'].isna().sum()} rows")
    return df


# QC: Sanity-Check Computed pIC50 Against ChEMBL's `pchembl_value`



In [ ]:
def sanity_check_pIC50_vs_pchembl(df):
    check_df = df.dropna(subset=['pchembl_value']).copy()
    if len(check_df) == 0:
        print("  No rows with pchembl_value available -- skipping sanity check.")
        return None

    diff = (check_df['pIC50'] - check_df['pchembl_value']).abs()
    corr = check_df['pIC50'].corr(check_df['pchembl_value'])
    print(f"  n={len(check_df)} rows with both values | Pearson r={corr:.4f} | "
          f"mean abs diff={diff.mean():.3f} log units | max abs diff={diff.max():.3f}")

    large_mismatch = check_df[diff > 0.3]
    if len(large_mismatch) > 0:
        print(f"  [warn] {len(large_mismatch)} rows differ from pchembl_value by >0.3 log units -- "
              f"worth a manual look (unit/assay annotation issue?).")
    else:
        print("  No rows differ from pchembl_value by more than 0.3 log units.")

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(check_df['pchembl_value'], check_df['pIC50'], s=10, alpha=0.4, color="#4C78A8")
    lims = [min(check_df['pchembl_value'].min(), check_df['pIC50'].min()) - 0.2,
            max(check_df['pchembl_value'].max(), check_df['pIC50'].max()) + 0.2]
    ax.plot(lims, lims, "k--", linewidth=1, label="y = x")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("ChEMBL pchembl_value")
    ax.set_ylabel("Computed pIC50")
    ax.set_title(f"pIC50 sanity check (r={corr:.3f}, n={len(check_df)})")
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    plt.savefig("figure_00_pic50_sanity_check.png", bbox_inches="tight")
    plt.show()
    return corr


# Step 5: Assay Confidence Filtering (`confidence_score >= 8`)




In [ ]:
def fetch_assay_confidence(assay_ids):
    '''Look up ChEMBL's per-assay confidence_score (0-9) for a list of assay_chembl_ids.
    confidence_score lives on the `assay` resource, not `activity`, so this is
    a separate lookup, batched to keep each request URL a reasonable length.'''
    assay_client = new_client.assay
    unique_ids = sorted({a for a in assay_ids if isinstance(a, str) and a})

    records = []
    chunk_size = 50
    for i in range(0, len(unique_ids), chunk_size):
        chunk = unique_ids[i:i + chunk_size]
        res = assay_client.filter(assay_chembl_id__in=chunk).only(
            ['assay_chembl_id', 'confidence_score']
        )
        records.extend(list(res))

    conf_df = pd.DataFrame.from_records(records, columns=['assay_chembl_id', 'confidence_score'])
    conf_df['confidence_score'] = pd.to_numeric(conf_df['confidence_score'], errors='coerce')
    return conf_df.drop_duplicates(subset='assay_chembl_id')


def assay_confidence_filter(df, min_confidence=MIN_ASSAY_CONFIDENCE):
    before = len(df)
    conf_df = fetch_assay_confidence(df['assay_chembl_id'])
    df = df.merge(conf_df, on='assay_chembl_id', how='left')

    n_unmatched = df['confidence_score'].isna().sum()
    print(f"  Confidence scores retrieved for {conf_df['assay_chembl_id'].nunique()} unique assays "
          f"({n_unmatched} rows had no matching assay record and will be dropped)")

    counts = df['confidence_score'].value_counts(dropna=False).sort_index()
    labels = [("NA" if pd.isna(s) else str(int(s))) for s in counts.index]
    colors = ["#4C78A8" if (pd.notna(s) and s >= min_confidence) else "#E45756" for s in counts.index]

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(labels, counts.values, color=colors)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v, str(v), ha="center", va="bottom", fontsize=8)
    ax.set_xlabel("Assay confidence_score")
    ax.set_ylabel("Activity records")
    ax.set_title(f"Assay confidence score distribution (kept: blue, >= {min_confidence})")
    plt.tight_layout()
    plt.savefig("figure_01_assay_confidence.png", bbox_inches="tight")
    plt.show()

    df = df[df['confidence_score'] >= min_confidence].copy()
    print(f"  After assay confidence filter (>= {min_confidence}): {len(df)} rows "
          f"(dropped {before - len(df)})")
    return df


# Step 6: Check ChEMBL `data_validity_comment` (Very Useful)




In [ ]:
def check_data_validity(df):
    before = len(df)
    df = df.copy()
    df['data_validity_comment'] = df['data_validity_comment'].fillna('(clean / no comment)')
    counts = df['data_validity_comment'].value_counts().sort_values()

    print("  data_validity_comment breakdown:")
    print(counts.to_string())

    colors = ["#59A14F" if c == '(clean / no comment)' else "#E45756" for c in counts.index]
    fig, ax = plt.subplots(figsize=(7, max(3, 0.4 * len(counts))))
    ax.barh(counts.index, counts.values, color=colors)
    for i, v in enumerate(counts.values):
        ax.text(v, i, f"  {v}", va="center", fontsize=8)
    ax.set_xlabel("Activity records")
    ax.set_title("ChEMBL data_validity_comment breakdown (green = kept)")
    plt.tight_layout()
    plt.savefig("figure_02_data_validity_comments.png", bbox_inches="tight")
    plt.show()

    df = df[df['data_validity_comment'] == '(clean / no comment)'].drop(columns=['data_validity_comment'])
    print(f"  After data_validity_comment filter: {len(df)} rows (dropped {before - len(df)} flagged records)")
    return df


# Step 7: Validate SMILES & Standardize Structures



In [ ]:
def validate_smiles(df):
    def to_standard_mol(smi):
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                return None
            mol = _largest_fragment_remover.choose(mol)  # drop counter-ion(s)
            mol = _uncharger.uncharge(mol)                # neutralize what remains
            return mol
        except Exception:
            return None

    df = df.copy()
    df['mol'] = df['canonical_smiles'].apply(to_standard_mol)
    before = len(df)
    df = df[df['mol'].notna()].copy()

    # Overwrite canonical_smiles with the STANDARDIZED structure's canonical
    # SMILES -- not the raw ChEMBL string -- so every downstream artifact
    # (this CSV, and Notebook 1's fingerprints/descriptors) refers to the
    # same molecule that InChIKey dedup and Lipinski filtering below use.
    df['canonical_smiles'] = df['mol'].apply(Chem.MolToSmiles)

    print(f"  After SMILES validation + standardization: {len(df)} rows "
          f"(dropped {before - len(df)} invalid/unparseable)")
    return df


# Step 8: Deduplicate by InChIKey (Median pIC50 Across Repeats)



In [ ]:
def dedup_by_inchikey(df):
    def get_inchikey(mol):
        try:
            return inchi.MolToInchiKey(mol)
        except Exception:
            return None

    df = df.copy()
    df['inchikey'] = df['mol'].apply(get_inchikey)
    before = len(df)
    df = df.dropna(subset=['inchikey'])

    df['pIC50'] = df.groupby('inchikey')['pIC50'].transform('median')
    df = df.sort_values('molecule_chembl_id').reset_index(drop=True)
    df = df.drop_duplicates(subset='inchikey', keep='first').reset_index(drop=True)

    print(f"  After InChIKey dedup (median pIC50 across repeats): {len(df)} rows "
          f"(dropped {before - len(df)} duplicate/repeat measurements)")
    return df


# Step 9: Filter Out PAINS / Nuisance Compounds (Highly Recommended)



In [ ]:
def pains_filter(df):
    before = len(df)

    def get_pains_alert(mol):
        entry = _pains_catalog.GetFirstMatch(mol)
        return entry.GetDescription() if entry is not None else None

    df = df.copy()
    df['pains_alert'] = df['mol'].apply(get_pains_alert)
    n_flagged = df['pains_alert'].notna().sum()
    print(f"  PAINS/nuisance alerts found in {n_flagged} of {len(df)} compounds")

    if n_flagged > 0:
        top_alerts = df.loc[df['pains_alert'].notna(), 'pains_alert'].value_counts().head(15)
        fig, ax = plt.subplots(figsize=(7, max(3, 0.35 * len(top_alerts))))
        ax.barh(top_alerts.index[::-1], top_alerts.values[::-1], color="#E45756")
        for i, v in enumerate(top_alerts.values[::-1]):
            ax.text(v, i, f"  {v}", va="center", fontsize=8)
        ax.set_xlabel("Compounds flagged")
        ax.set_title(f"Top PAINS/nuisance alerts ({n_flagged}/{len(df)} compounds flagged)")
        plt.tight_layout()
        plt.savefig("figure_03_pains_alerts.png", bbox_inches="tight")
        plt.show()
    else:
        print("  No PAINS/nuisance alerts matched -- skipping alert-frequency plot.")

    df = df[df['pains_alert'].isna()].drop(columns=['pains_alert'])
    print(f"  After PAINS/nuisance compound filter: {len(df)} rows (dropped {before - len(df)} flagged compounds)")
    return df


# Step 10: Outlier Analysis on pIC50



In [ ]:
def analyze_pIC50_outliers(df, iqr_multiplier=PIC50_IQR_MULTIPLIER, remove=REMOVE_PIC50_OUTLIERS):
    df = df.copy()
    q1, q3 = df['pIC50'].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - iqr_multiplier * iqr, q3 + iqr_multiplier * iqr

    df['pIC50_outlier'] = (df['pIC50'] < lower) | (df['pIC50'] > upper)
    n_out = int(df['pIC50_outlier'].sum())
    print(f"  pIC50 IQR bounds (k={iqr_multiplier}): [{lower:.2f}, {upper:.2f}]")
    print(f"  {n_out} of {len(df)} compounds ({100 * n_out / len(df):.1f}%) fall outside this range")

    if n_out:
        df.loc[df['pIC50_outlier']].sort_values('pIC50').to_csv("AKT1_pIC50_outliers.csv", index=False)
        print("  Flagged rows written to AKT1_pIC50_outliers.csv for manual review "
              "(not auto-dropped -- see markdown above).")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].boxplot(df['pIC50'], patch_artist=True,
                     boxprops=dict(facecolor="#4C78A8", alpha=0.6),
                     medianprops=dict(color="black"))
    axes[0].set_ylabel("pIC50")
    axes[0].set_xticks([])
    axes[0].set_title("pIC50 boxplot")

    inliers = df.loc[~df['pIC50_outlier'], 'pIC50']
    outliers = df.loc[df['pIC50_outlier'], 'pIC50']
    axes[1].hist(inliers, bins=30, color="#4C78A8", alpha=0.85, label=f"inlier (n={len(inliers)})")
    if n_out:
        axes[1].hist(outliers, bins=30, color="#E45756", alpha=0.85, label=f"outlier (n={len(outliers)})")
    axes[1].axvline(lower, color="black", linestyle="--", linewidth=1)
    axes[1].axvline(upper, color="black", linestyle="--", linewidth=1)
    axes[1].set_xlabel("pIC50")
    axes[1].set_ylabel("Count")
    axes[1].set_title("pIC50 distribution vs. IQR bounds")
    axes[1].legend(frameon=False, fontsize=8)

    plt.tight_layout()
    plt.savefig("figure_04_pIC50_outliers.png", bbox_inches="tight")
    plt.show()

    if remove and n_out:
        before = len(df)
        df = df[~df['pIC50_outlier']].copy()
        print(f"  REMOVE_PIC50_OUTLIERS=True -> dropped {before - len(df)} outlier rows from the dataset")

    return df


# Step 11: Lipinski Rule-of-Five Filter



In [ ]:
def lipinski_filter(df):
    df = df.copy()
    df['MW']   = df['mol'].apply(Descriptors.MolWt)
    df['LogP'] = df['mol'].apply(Descriptors.MolLogP)
    df['HBD']  = df['mol'].apply(Descriptors.NumHDonors)
    df['HBA']  = df['mol'].apply(Descriptors.NumHAcceptors)

    before = len(df)
    df['ro5_violations'] = (
        (df['MW']   > 500).astype(int) +
        (df['LogP'] > 5).astype(int)   +
        (df['HBD']  > 5).astype(int)   +
        (df['HBA']  > 10).astype(int)
    )
    df = df[df['ro5_violations'] <= 1]
    print(f"  After Lipinski filter: {len(df)} rows (dropped {before - len(df)} violators)")
    return df


# Step 12: Compute Remaining Descriptors


In [ ]:
def compute_descriptors(df):
    df = df.copy()
    df['TPSA']       = df['mol'].apply(Descriptors.TPSA)
    df['RotBonds']   = df['mol'].apply(Descriptors.NumRotatableBonds)
    df['HeavyAtoms'] = df['mol'].apply(Descriptors.HeavyAtomCount)
    print(f"  Descriptors computed: MW, LogP, HBD, HBA, TPSA, RotBonds, HeavyAtoms")
    return df


# Master Pipeline


In [ ]:
def run_pipeline(chembl_id, target_name, output_csv):
    attrition = []

    def record(step_label, df):
        attrition.append({"step": step_label, "n_rows": len(df)})
        return df

    print(f"\nFetching {target_name} ({chembl_id})...")
    df = fetch_bioactivity(chembl_id, target_name)
    print(f"  Raw records: {len(df)}")
    record("1. Raw ChEMBL records", df)

    print("\nStep 2: Drop missing SMILES / IC50")
    df = drop_missing(df)
    record("2. After drop missing", df)

    print("\nStep 3: Normalize units to nM")
    df = normalize_units(df)
    record("3. After unit normalization", df)

    print("\nStep 4: Compute pIC50")
    df = compute_pIC50(df)
    record("4. After pIC50 computation", df)

    print("\nQC: computed pIC50 vs ChEMBL pchembl_value")
    sanity_check_pIC50_vs_pchembl(df)

    print(f"\nStep 5: Assay confidence filtering (>= {MIN_ASSAY_CONFIDENCE})")
    df = assay_confidence_filter(df)
    record("5. After assay confidence filter", df)

    print("\nStep 6: Check data_validity_comment")
    df = check_data_validity(df)
    record("6. After data_validity_comment check", df)

    print("\nStep 7: Validate SMILES & standardize structures")
    df = validate_smiles(df)
    record("7. After standardization", df)

    print("\nStep 8: Deduplicate by InChIKey (median pIC50 across repeats)")
    df = dedup_by_inchikey(df)
    record("8. After InChIKey dedup", df)

    print("\nStep 9: PAINS / nuisance compound filter")
    df = pains_filter(df)
    record("9. After PAINS/nuisance filter", df)

    print("\nStep 10: Outlier analysis on pIC50")
    df = analyze_pIC50_outliers(df)
    outlier_label = "10. After pIC50 outlier analysis" + (" (removed)" if REMOVE_PIC50_OUTLIERS else " (flagged only)")
    record(outlier_label, df)

    print("\nStep 11: Lipinski Ro5 filter")
    df = lipinski_filter(df)
    record("11. After Lipinski filter", df)

    print("\nStep 12: Compute remaining descriptors")
    df = compute_descriptors(df)
    record("12. After descriptors (final)", df)

    df = df.drop(columns=['mol']).reset_index(drop=True)
    df.to_csv(output_csv, index=False)
    print(f"\nSaved: {output_csv} | Final compounds: {len(df)}")

    # --- Attrition table + funnel chart ---
    attrition_df = pd.DataFrame(attrition)
    prev = attrition_df["n_rows"].shift(1)
    prev.iloc[0] = attrition_df["n_rows"].iloc[0]
    attrition_df["dropped"] = (prev - attrition_df["n_rows"]).astype(int)
    attrition_df.to_csv("data_curation_attrition.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(attrition_df["step"][::-1], attrition_df["n_rows"][::-1], color="#4C78A8")
    for i, (n, d) in enumerate(zip(attrition_df["n_rows"][::-1], attrition_df["dropped"][::-1])):
        label = f"{n}" if d == 0 else f"{n}  (-{d})"
        ax.text(n, i, f"  {label}", va="center", fontsize=8)
    ax.set_xlabel("Compounds remaining")
    ax.set_title(f"{target_name} data curation attrition")
    plt.tight_layout()
    plt.savefig("figure_00_data_attrition.png", bbox_inches="tight")
    plt.show()

    print("\nAttrition summary:")
    print(attrition_df.to_string(index=False))

    print("\nFinal dataset summary:")
    summary_cols = ['molecule_chembl_id', 'IC50_nM', 'pIC50', 'confidence_score',
                     'MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'ro5_violations']
    print(df[[c for c in summary_cols if c in df.columns]].describe().round(2))

    return df, attrition_df


# Run


In [ ]:
AKT1_df, AKT1_attrition = run_pipeline('CHEMBL4282', 'AKT1', 'AKT1_clean.csv')
